In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Deterministic and Bayesian Refinement: LBCO, HRPT

This tutorial demonstrates a practical two-stage workflow for powder
diffraction analysis with EasyDiffraction.

In the first stage, we run a fast local refinement to obtain a sensible
point estimate and parameter uncertainties. In the second stage, we use
these refined values to define fit bounds and then sample the posterior
distribution with DREAM.

The example uses constant-wavelength neutron powder diffraction data
for La0.5Ba0.5CoO3 measured on HRPT at PSI.

The goal is not only to obtain a good fit, but also to answer Bayesian
questions such as:

- Which parameter values are most probable?
- How broad are the credible intervals?
- Which parameters are strongly correlated?
- How much uncertainty propagates into the calculated diffraction
  pattern?

## Import Library

In [2]:
import easydiffraction as ed

## Step 1: Create a Project Container

The project object keeps structures, experiments, fit settings, and
plotting utilities together in a single place. We will build the full
workflow inside this object.

In [3]:
project = ed.Project()

## Step 2: Build the Structural Model

We define a simple cubic perovskite model for LBCO. La and Ba share the
same crystallographic site with equal occupancy, while Co and O occupy
the remaining ideal perovskite positions.

In [4]:
project.structures.create(name='lbco')

In [5]:
structure = project.structures['lbco']

In [6]:
structure.space_group.name_h_m = 'P m -3 m'
structure.space_group.it_coordinate_system_code = '1'

In [7]:
structure.cell.length_a = 3.88

The atom-site definitions below form the starting structural model. The
parameters are intentionally reasonable rather than fully optimized,
because the refinement step will improve them.

In [8]:
structure.atom_sites.create(
    label='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    adp_type='Biso',
    adp_iso=0.2190,
)
structure.atom_sites.create(
    label='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='c',
    adp_type='Biso',
    adp_iso=1.3916,
)

## Step 3: Define the Diffraction Experiment

Next we download the measured powder pattern, create a neutron powder
experiment, and configure the instrument, profile, background, and
excluded regions.

#### Download the Measured Data

In [9]:
data_path = ed.download_data(id=3, destination='data')

Getting data...
Data #3: La0.5Ba0.5CoO3, HRPT (PSI), 300 K
✅ Data #3 already present at 'data/ed-3.xye'. Keeping existing file.


#### Create the Experiment Object

In [10]:
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully
Experiment 🔬 'hrpt'. Number of data points: 3098.


In [11]:
experiment = project.experiments['hrpt']

#### Set Instrument and Peak-Profile Parameters

These values provide the initial instrument description for the local
refinement. Later, a subset of them will be refined.

In [12]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.0

In [13]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1204
experiment.peak.broad_lorentz_y = 0.0844

#### Add Background Points and Excluded Regions

The line-segment background is defined by a few anchor points. We also
exclude regions that are not intended to contribute to the fit.

In [14]:
experiment.background.create(id='1', x=10, y=168.5585)
experiment.background.create(id='2', x=30, y=164.3357)
experiment.background.create(id='3', x=50, y=166.8881)
experiment.background.create(id='4', x=110, y=175.4006)
experiment.background.create(id='5', x=165, y=174.2813)

In [15]:
experiment.excluded_regions.create(id='1', start=0, end=30)
experiment.excluded_regions.create(id='2', start=70, end=180)

#### Link the Structural Phase to the Experiment

In [16]:
experiment.linked_phases.create(id='lbco', scale=9.1351)

## Step 4: Run an Initial Local Refinement

Before Bayesian sampling, it is useful to run a deterministic fit. This
gives us:

- a good point estimate near the best-fit region,
- uncertainties from the local optimizer,
- a quick check that the model and experiment are configured
  sensibly.

In this tutorial we refine only a small set of parameters that are easy
to interpret in the later Bayesian stage.

In [17]:
structure.cell.length_a.free = True
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.instrument.calib_twotheta_offset.free = True

We choose the BUMPS Levenberg-Marquardt minimizer as a fast local
optimizer. Its main purpose here is to provide a stable starting point
and uncertainty estimates for the Bayesian run.

In [18]:
project.analysis.fit.show_minimizer_types()
project.analysis.fit.minimizer_type = 'bumps (lm)'

Minimizer types


,,Type,Description
1,,bumps,Bumps library using the default Levenberg-Marquardt method
2,,bumps (amoeba),Bumps library with Nelder-Mead simplex method
3,,bumps (de),Bumps library with differential evolution method
4,,bumps (dream),Bumps library with DREAM Bayesian sampling
5,,bumps (lm),Bumps library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,lmfit,LMFIT library using the default Levenberg-Marquardt least squares method
8,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
9,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


Current minimizer changed to
bumps (lm)


In [19]:
project.analysis.fit()

Standard fitting
📋 Using experiment 🔬 'hrpt' for 'single' fitting
🚀 Starting fit process with 'bumps (lm)'...
📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.14,736.04,
2,7,0.25,315.22,57.2% ↓
3,12,0.34,93.97,70.2% ↓
4,17,0.43,40.33,57.1% ↓
5,21,0.51,39.06,3.1% ↓
6,22,0.54,28.50,27.0% ↓
7,26,0.61,24.69,13.4% ↓
8,27,0.63,16.23,34.2% ↓
9,31,0.71,14.77,9.0% ↓
10,32,0.73,2.74,81.4% ↓


🏆 Best goodness-of-fit (reduced χ²) is 1.37 at iteration 68
✅ Fitting complete.


In [20]:
project.analysis.display.fit_results()

Fit results
✅ Success: True
⏱️ Fitting time: 1.39 seconds
📏 Goodness-of-fit (reduced χ²): 1.37
📏 R-factor (Rf): 5.27%
📏 R-factor squared (Rf²): 4.42%
📏 Weighted R-factor (wR): 3.78%
📈 Fitted parameters:


,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,lbco,cell,,length_a,3.8800,3.8907,0.0002,Å,0.27 % ↑
2,hrpt,peak,,broad_gauss_u,0.1000,0.0333,0.0163,deg²,66.71 % ↓
3,hrpt,peak,,broad_gauss_v,-0.1000,-0.0934,0.0091,deg²,6.57 % ↓
4,hrpt,instrument,,twotheta_offset,0.0000,0.6221,0.0029,deg,N/A


The correlation plot shows how strongly the fitted parameters move
together in the local refinement. The measured-vs-calculated plots show
how well the refined model reproduces the data globally and in a zoomed
region.

In [21]:
project.display.plotter.plot_param_correlations(show_diagonal=True)

In [22]:
project.display.plotter.plot_meas_vs_calc(expt_name='hrpt')

In [23]:
project.display.plotter.plot_meas_vs_calc(expt_name='hrpt', x_min=65, x_max=68)

## Step 5: Prepare for Bayesian Sampling

DREAM requires finite bounds for the free parameters. Instead of
setting them manually, we derive them from the uncertainties estimated
in the local refinement.

The helper method `set_fit_bounds_from_uncertainty` centers the bounds
on the current parameter value and expands them by a chosen multiple of
the reported uncertainty.

In [24]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lbco,cell,,length_a,3.89066,0.00021,-inf,inf,Å
2,hrpt,peak,,broad_gauss_u,0.03329,0.01633,-inf,inf,deg²
3,hrpt,peak,,broad_gauss_v,-0.09343,0.00911,-inf,inf,deg²
4,hrpt,instrument,,twotheta_offset,0.62207,0.00294,-inf,inf,deg


In [25]:
structure.cell.length_a.set_fit_bounds_from_uncertainty(multiplier=4)
experiment.peak.broad_gauss_u.set_fit_bounds_from_uncertainty(multiplier=4)
experiment.peak.broad_gauss_v.set_fit_bounds_from_uncertainty(multiplier=4)
experiment.instrument.calib_twotheta_offset.set_fit_bounds_from_uncertainty(multiplier=4)

Displaying the free parameters again is a convenient way to confirm
that the fit bounds have been assigned as expected before launching the
sampler.

In [26]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lbco,cell,,length_a,3.89066,0.00021,3.88983,3.89148,Å
2,hrpt,peak,,broad_gauss_u,0.03329,0.01633,-0.03205,0.09863,deg²
3,hrpt,peak,,broad_gauss_v,-0.09343,0.00911,-0.12988,-0.05699,deg²
4,hrpt,instrument,,twotheta_offset,0.62207,0.00294,0.61031,0.63383,deg


## Step 6: Configure and Run DREAM

We now switch from the local minimizer to the Bayesian DREAM sampler.

The settings below are intentionally small so the tutorial runs
quickly. For production analysis you would usually increase the number
of steps and often the burn-in as well. When needed, the DREAM API
also lets you tune how chains are initialized through the `init`
setting.

In [27]:
project.analysis.fit.show_minimizer_types()

Minimizer types


,,Type,Description
1,,bumps,Bumps library using the default Levenberg-Marquardt method
2,,bumps (amoeba),Bumps library with Nelder-Mead simplex method
3,,bumps (de),Bumps library with differential evolution method
4,,bumps (dream),Bumps library with DREAM Bayesian sampling
5,*,bumps (lm),Bumps library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,lmfit,LMFIT library using the default Levenberg-Marquardt least squares method
8,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
9,,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [28]:
project.analysis.fit.minimizer_type = 'bumps (dream)'

Current minimizer changed to
bumps (dream)


In [29]:
project.analysis.fit.minimizer.steps = 2000  # 1000
project.analysis.fit.minimizer.burn = 400  # 200
project.analysis.fit.minimizer.thin = 1
project.analysis.fit.minimizer.pop = 4

In [ ]:
project.analysis.fit()

Standard fitting
📋 Using experiment 🔬 'hrpt' for 'single' fitting
🚀 Starting fit process with 'bumps (dream)'...
📈 Bayesian sampling progress:


,iteration,progress,time (s),log posterior,phase
1,1/2401,0.0%,0.30,-543.37,burn-in
2,101/2401,4.2%,27.35,-545.75,burn-in
3,200/2401,8.3%,53.82,-545.34,burn-in
4,300/2401,12.5%,80.18,-545.22,burn-in
5,400/2401,16.7%,106.78,-545.39,burn-in
6,401/2401,16.7%,107.05,-545.35,sampling
7,506/2401,21.1%,134.74,-545.28,sampling
8,612/2401,25.5%,164.08,-545.50,sampling
9,717/2401,29.9%,192.06,-546.77,sampling


## Step 7: Inspect Bayesian Results

The fit-results display now includes sampler settings, convergence
diagnostics, committed parameter values, and posterior summary
statistics.

In [ ]:
project.analysis.display.fit_results()

The correlation and posterior-pair plots are complementary:

- `plot_param_correlations` summarizes pairwise structure in a compact
  matrix.
- `plot_posterior_pairs` shows marginal densities on the diagonal and
  posterior contours off-diagonal.

In [ ]:
project.display.plotter.plot_param_correlations(show_diagonal=True)

In [ ]:
project.display.plotter.plot_posterior_pairs()

The one-dimensional posterior distributions below make it easier to
inspect individual parameters in isolation, including asymmetry or
multimodality.

In [ ]:
project.display.plotter.plot_param_distribution(structure.cell.length_a)
project.display.plotter.plot_param_distribution(experiment.peak.broad_gauss_u)
project.display.plotter.plot_param_distribution(experiment.peak.broad_gauss_v)
project.display.plotter.plot_param_distribution(experiment.instrument.calib_twotheta_offset)

Finally, the posterior predictive plot propagates the sampled parameter
uncertainty into the calculated diffraction pattern. Comparing this to
the zoomed measured-vs-calculated view helps assess whether the sampled
model family explains the data in the region of interest.

In [ ]:
project.display.plotter.plot_posterior_predictive(expt_name='hrpt')

A final zoomed measured-vs-calculated plot is useful for checking how
the posterior-supported model behaves in a narrow region of the pattern
after the Bayesian run.

In [ ]:
project.display.plotter.plot_meas_vs_calc(expt_name='hrpt', x_min=65, x_max=68)